In [ ]:
%pip install groq

In [123]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
from datetime import date
from dateutil.relativedelta import relativedelta

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

system_prompt = """You are an assistant that extracts structured information from user queries about company performance metrics.

Extract and return a JSON list of objects with the following fields from the user query:
- entity: Company name (like Amazon, Google, Netflix etc.)
- parameter: Performance metric (like GMV, revenue, profit, etc.)
- start_date: Start date mentioned (or empty string if not specified)
- end_date: End date mentioned (or empty string if not specified)

Choose the date thats smaller as the start date and the larger as the end date. Date should be in ISO format (YYYY-MM-DD).
Auto capitalise and fix case of the company names and performance metrics.
Identify any abbreviations and expand them (e.g., GMV to Gross Merchandise Value).

If the query includes multiple companies or asks for a comparison, return one JSON object per company in the list.

Respond only with valid JSON.
"""


In [124]:
query_input = input("Enter your query: ").strip()

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query_input}
    ]
)

llm_response = completion.choices[0].message.content


try:
    extracted_data = json.loads(llm_response)
except json.JSONDecodeError:
    exit(1)

In [ ]:
today = date.today()
one_year_ago = today - relativedelta(years=1)

history_file = "history.json"

if os.path.exists(history_file):
    try:
        with open(history_file, "r") as f:
            history = json.load(f)
            if "history" not in history:
                    history["history"] = []
    except json.JSONDecodeError:
        history = {"history": []}
else:
    with open(history_file, "w") as f:
        history = {"history": []}

for i in range(len(extracted_data)):
    history["history"].append(extracted_data[i])
    if(extracted_data[i]["start_date"] == ""):
        extracted_data[i]["start_date"] = one_year_ago.isoformat()
    if(extracted_data[i]["end_date"] == ""):
        extracted_data[i]["end_date"] = today.isoformat()
    if(len(history["history"]) > 6):
        history["history"].pop(0)

    print(extracted_data[i])

with open(history_file, "w") as f:
    json.dump(history, f, indent=2)

print("✅ History updated in history.json")